# Data Pipeline

Goal: turn a streaming text/code dataset into fixed-length `(x, y)` token blocks for causal language model training.


## Setup

Pick the best available PyTorch device and load `HF_TOKEN` from `.env`. The token stays outside the notebook.


In [37]:
import os

import torch
from dotenv import load_dotenv

load_dotenv()
access_token = os.environ.get("HF_TOKEN")
if access_token is None:
    raise ValueError("Set HF_TOKEN in your environment or in a local .env file.")

if torch.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

device


device(type='mps')

## Raw Streaming Dataset

`dataset` is a Hugging Face streaming dataset. It yields rows shaped like `{"text": ...}`. We interleave several sources into one raw text stream.


In [38]:
from datasets import interleave_datasets, load_dataset

def content_to_text(x):
    return {"text": x["content"]}

def stack_language(language):
    return load_dataset(
        "bigcode/starcoderdata",
        data_dir=language,
        streaming=True,
        split="train",
        token=access_token,
    ).select_columns(["content"]).map(content_to_text, remove_columns=["content"])

python_data = stack_language("python")
typescript_data = stack_language("typescript")
rust_data = stack_language("rust")
sql_data = stack_language("sql")
shell_data = stack_language("shell")

fineweb_edu = load_dataset(
    "HuggingFaceTB/smollm-corpus",
    "fineweb-edu-dedup",
    streaming=True,
    split="train",
    token=access_token,
).select_columns(["text"])

finemath = load_dataset(
    "HuggingFaceTB/finemath",
    "finemath-4plus",
    streaming=True,
    split="train",
    token=access_token,
).select_columns(["text"])

dataset = interleave_datasets(
    [python_data, finemath, fineweb_edu, typescript_data, rust_data, sql_data, shell_data],
    probabilities=[0.50, 0.18, 0.12, 0.08, 0.05, 0.04, 0.03],
    seed=42,
    stopping_strategy="first_exhausted",
)

print("Datasets loaded. Sample keys:", next(iter(dataset)).keys())


Datasets loaded. Sample keys: dict_keys(['text'])


## Tokenizer

The pretrained tokenizer maps text to integer token IDs. For this data pipeline we use `encode`, then append `eos_token_id` after each original example to mark document boundaries.


In [39]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bigcode/starcoder2-3b")

if tokenizer.eos_token_id is None:
    raise ValueError("Tokenizer must define eos_token_id for document boundaries.")

sample = "def hello_world():"
print("Tokens:", tokenizer.tokenize(sample))
print("Token IDs:", tokenizer.encode(sample))
print("Decoded:", tokenizer.decode(tokenizer.encode(sample)))
print("EOS:", tokenizer.eos_token, tokenizer.eos_token_id)


Tokens: ['def', 'Ġhello', '_', 'world', '():']
Token IDs: [610, 17966, 100, 5879, 2284]
Decoded: def hello_world():
EOS: <|endoftext|> 0


## Training Dataset

`CoderDataset` wraps the raw stream. It keeps a small token buffer, appends EOS after every source example, and yields fixed-length `(x, y)` tensors. `y` is `x` shifted one token forward.


In [40]:
from torch.utils.data import IterableDataset

class CoderDataset(IterableDataset):
    def __init__(self, dataset, tokenizer, context_length):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.context_length = context_length
        self.eos_token_id = tokenizer.eos_token_id

        if self.eos_token_id is None:
            raise ValueError("Tokenizer must define eos_token_id.")

    def __iter__(self):
        buffer = []

        for example in self.dataset:
            encoded = self.tokenizer.encode(example["text"])
            encoded.append(self.eos_token_id)
            buffer.extend(encoded)

            while len(buffer) >= self.context_length + 1:
                x = torch.tensor(buffer[0 : self.context_length], dtype=torch.long)
                y = torch.tensor(buffer[1 : self.context_length + 1], dtype=torch.long)
                buffer = buffer[self.context_length:]
                yield x, y


## Smoke Test

Use a tiny streamed subset first. The checks prove that one training sample has the expected shape, dtype, and next-token shift.


In [41]:
context_length = 1024
small_dataset = dataset.take(10)
coder_dataset = CoderDataset(small_dataset, tokenizer, context_length)

x, y = next(iter(coder_dataset))

print("x shape:", x.shape)
print("y shape:", y.shape)
print("x dtype:", x.dtype)
print("y dtype:", y.dtype)
print("shift check:", torch.equal(x[1:], y[:-1]))
print("x first tokens:", x[:5])
print("y first tokens:", y[:5])


x shape: torch.Size([1024])
y shape: torch.Size([1024])
x dtype: torch.int64
y dtype: torch.int64
shift check: True
x first tokens: tensor([2042,   51,  244,   55,   57])
y first tokens: tensor([ 51, 244,  55,  57,  64])


## DataLoader

The dataset yields one sample at a time. `DataLoader` batches those samples into tensors shaped `[batch_size, context_length]`.


In [42]:
from torch.utils.data import DataLoader

batch_size = 2
train_loader = DataLoader(
    coder_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

x_batch, y_batch = next(iter(train_loader))

print("x batch shape:", x_batch.shape)
print("y batch shape:", y_batch.shape)
print("batch shift check:", torch.equal(x_batch[:, 1:], y_batch[:, :-1]))
print("first sample x:", x_batch[0, :5])
print("first sample y:", y_batch[0, :5])


x batch shape: torch.Size([2, 1024])
y batch shape: torch.Size([2, 1024])
batch shift check: True
first sample x: tensor([2042,   51,  244,   55,   57])
first sample y: tensor([ 51, 244,  55,  57,  64])
